In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [ ]:
# Data: ToTensor scales to [0, 1]; the model flattens the images itself
train_data = datasets.MNIST("mnist_data", train=True, transform=ToTensor(), download=True)
test_data = datasets.MNIST("mnist_data", train=False, transform=ToTensor())
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

In [ ]:
# Model
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10),   # outputs logits; softmax is inside the loss
)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [ ]:
# Training
for epoch in range(10):
    for X, y in train_loader:
        loss = loss_fn(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch}, last batch loss = {loss.item():.4f}")

In [ ]:
# Evaluation
model.eval()  # Switches off Dropout and Batch Normalization, if used
correct = 0
with torch.no_grad():
    for X, y in test_loader:
        correct += (model(X).argmax(dim=1) == y).sum().item()
print(f"Accuracy = {correct / len(test_data):.4f}")